# Generate synthetic data for the agnews dataset

- Llama-2-7b-hf  
  1. baseline
  2. targeted + linguistic tags
  3. unsupervised context
  4. (unsupervised context + linguistic tags)

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, LlamaForCausalLM, BitsAndBytesConfig

# CHANGE WORKING DIRECTORY TO ROOT
current_dir = os.path.basename(os.getcwd())
if current_dir == "src":
    os.chdir("..")
elif os.path.basename(os.getcwd()) == "bai-thesis-nlp":  
    pass
else:
    os.chdir("../..")
from src._utils._generate_dataset import main_generate_dataset
from src._utils._helpers import get_generated_examples_df, clear_cuda_cache

# get true labels
df_real = pd.read_csv("real_data/train/agnewstrainAll.csv").rename(
    columns={"2": "text", "3": "label"}
)
correct_labels = df_real["label"].unique().tolist()
labels_str = ", ".join(correct_labels)
labels_str_bullet = "\n".join([f"- {name}" for name in correct_labels])
model = None
HF_TOKEN = open("src/_utils/hf_token.txt","r").read() # your huggingface token

In [2]:
# chat tamplate format:
# <s>[INST] <<SYS>>
# {{ system_prompt }}
# <</SYS>>
# 
# {{ user_msg_1 }} [/INST] {{ model_answer_1 }} </s><s>[INST] {{ user_msg_2 }} [/INST]

PROMPTS = {}
system = "You are an expert in journalism and NLP specializing in news classification."
assistant_response = "Of course! I'm happy to help. Please provide me with the category you would like me to focus on, and I will generate a high-quality short document for you."


prompt_baseline = f"""\
Your task is to generate one high-quality short document (around 30 words), that talks about one of the following four News categories:  
{labels_str_bullet}

Choose one of the categories (labels), generate the corresponding text and return it in the following JSON format:

```json
{{
    "text": "<text of the document>", 
    "label": "<corresponding label>", 
}}
```
"""
PROMPTS["baseline"] = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt_baseline},
    {"role": "assistant", "content": assistant_response},
    # {"role": "user", "content": f"label: {random_label}"},
]


prompt_targeted =  f"""\
Your task is to generate one high-quality short document (around 30 words), that talks about one of the following four News categories:  
{labels_str_bullet}

For each example, also list the key phenomena it covers.

### **Follow these topics:**
- **Business**  
  - Markets  
  - Economy  
  - Companies  
  - Startups  
  - Regulations  

- **Sci/Tech**  
  - AI  
  - Space  
  - Cybersecurity  
  - Biotech  
  - Climate  

- **Sports**  
  - Events  
  - Records  
  - Highlights  
  - Scandals  
  - Olympics  

- **World**  
  - Politics  
  - Conflicts  
  - Disasters  
  - Human Rights  
  - Trade

### **Output Format (JSON)**
The labels must be one of the specified categories, which are: {labels_str}. \
Choose one of the categories (labels), generate the corresponding text, write the corresponding phenomena and return it in the following JSON format:

```json
{{
    "text": "<text of the document>", 
    "label": "<corresponding label>", 
    "phenomena": ["<phenomenon1>", "<phenomenon2>", ...]
}}
```
"""
PROMPTS["targeted + linguistic tags"] = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt_targeted},
    {"role": "assistant", "content": assistant_response},
    # {"role": "user", "content": f"label: {random_label}"},
]

PROMPTS["unsupervised context"] = (
# baseline prompt
"""\
You are an expert in journalism and NLP specialized in news classification. \
Your task is to generate an high-quality short documents, that talks about one of the following four News categories (labels):
- Business
- Sci/Tech
- Sports
- World.

Here some examples of the documents you can use as a reference:
""",
# postfix
"""
Generate a new news document, with the corresponding category (label) with the following format:
```json
[
    {
        "text": "<text of the document>", 
        "label": "<corresponding label>",
    }
]
```
"""
)

In [3]:
#### FUNCTION FOR TAKING CONTEXT EXAMPLES
# we sample randomly "num_examples_per_prompt" examples "num_prompts" times
# and store them in a list of lists

def get_context_examples(df, num_examples_per_prompt, num_prompts):
    context_examples = []
    for _ in range(num_prompts):
        examples = df.sample(n=num_examples_per_prompt, replace=False, random_state=np.random.randint(0, 1e6))
        context_examples.append(examples['text'].tolist())
    return context_examples

# Llama-2-7b-hf

In [4]:
#############################################
# LOAD MODEL
#############################################

if model:
    clear_cuda_cache(model)

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model_name = "meta-llama/Llama-2-7b-chat-hf"
model = LlamaForCausalLM.from_pretrained(
            model_name, 
            token=HF_TOKEN,
            torch_dtype=torch.float16,
            attn_implementation='flash_attention_2',
            quantization_config=quantization_config,
            low_cpu_mem_usage=True
        ).to("cuda")

tokenizer = LlamaTokenizer.from_pretrained(model_name, token=HF_TOKEN)

OUTPUT_DIR = "synthetic_data/datasets/Llama-2-7b-hf/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
base_config = {
    "dataset": "agnews",
    "model": model,
    "tokenizer": tokenizer,
    # "generation_method": "baseline",
    #### We are going to give as input the list of messages
    # "prompt": {system:..., user: prompt, assistant: None},
    "apply_chat_template": True,
    "num_examples": 500,
    "max_new_tokens": 1024,
    "seed": 42,
    #"json_output_file": OUTPUT_DIR+"syn_agnews_baseline_500.json",
    "log_file": OUTPUT_DIR+"generate_dataset_agnews_log.json",
    "correct_labels": correct_labels,
    "correct_fields": ["text", "label"],
    ### context
    # "context_examples": None,
    # "prompt_postfix": None,
    "verbose": False,
    ### At each iteration we ask randomly for a label
    "append_random_label": True, 
}

### 1. baseline

In [6]:
name = "baseline"
config = base_config.copy()
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+"syn_agnews_baseline_500.json"
main_generate_dataset(config)

Generating Examples:   0%|          | 0/5 [00:00<?, ?ex/s]

Generating Examples: 100%|██████████| 5/5 [00:19<00:00,  3.95s/ex, examples=5/5, run=6]

⏱️ Time taken: 19.77 seconds.


### 2. targeted + linguistic tags

In [7]:
config = base_config.copy()
name = "targeted + linguistic tags"
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+"syn_agnews_targeted+tags_500.json"
config["correct_fields"] = ["text", "label", "phenomena"]
main_generate_dataset(config)

Generating Examples: 100%|██████████| 5/5 [00:40<00:00,  8.04s/ex, examples=5/5, run=6]

⏱️ Time taken: 40.21 seconds.


In [ ]:
# ### Generate other 500 examples

# config['seed'] = config['seed']*8
# config['json_output_file'] = OUTPUT_DIR+"syn_agnews_targeted+tags_500_2.json"
# main_generate_dataset(config)

### 3. unsupervised context

In each prompt we attach n (5) examples sampled randomly from the train set. The samples are used without the labels, so they works as unsupervised context for the model, when we will generate the new synthetic sample. 

In [ ]:
# we take more than 500 because it can happen that some prompt
# generate the example in the wrong format, so is not read correctly (and discarded)
num_prompts = 1000
num_examples_per_prompt = 5
np.random.seed(42)
context_examples = get_context_examples(df_real, num_examples_per_prompt, num_prompts)
print(f"Number prompts: {len(context_examples)}")
print(f"Number of examples per prompt: {len(context_examples[0])}")
print(context_examples[0])

In [ ]:
config = base_config.copy()
name = "unsupervised context"
config["generation_method"] = name
config["prompt"] = PROMPTS[name][0]
config["prompt_postfix"] = PROMPTS[name][1]
config["max_new_tokens"] = 2048
config["json_output_file"] = OUTPUT_DIR+"syn_agnews_unsupervisedContext_500.json"
config["context_examples"] = context_examples
main_generate_dataset(config)

In [ ]:
generated_df, _ = get_generated_examples_df(OUTPUT_DIR+"syn_agnews_unsupervisedContext_500.json")

for i in range(5):
    print("TEXT: "+generated_df.iloc[i]['text'])
    print("LABEL: "+generated_df.iloc[i]['label'])
    print("CONTEXT EXAMPLES:")
    for j in range(len(generated_df.iloc[i]['context_examples'])):
        print("- "+generated_df.iloc[i]['context_examples'][j])

    print("\n"+"=="*50)

In [ ]:
# Unsupervised context + tags
# ...